# NakedAgent vs OntologyAgent — truck logistics benchmark

Side-by-side evaluation of two Fabric Data Agents on 18 long-haul trucking scenarios (sanity, multi-hop, graph, governed metrics, ambiguity, guardrails).

## Architecture — two different query engines

`scripts/05_setup_agents.py` wires the two agents so they each go through exactly one data surface:

| Agent | Data source | Query engine | What it "sees" |
|---|---|---|---|
| `NakedAgent` | Lakehouse Delta tables | Spark SQL | 11 raw tables with `_id` FK columns |
| `OntologyAgent` | Ontology (graph model) | GQL | 11 entity types + 19 typed relationships |

The ontology's graph model is populated by `scripts/04_refresh_and_validate.py`, which materializes nodes and edges from the Lakehouse tables via the bindings + contextualizations. Both agents see the *same underlying data*; they just traverse it through different engines.

## How the notebook works

For each scenario the notebook:

1. Sends the question to `NakedAgent` via `FabricOpenAI` and captures the final text reply
2. Does the same for `OntologyAgent`
3. Scores each answer with a deterministic token check — every `ontology_signals` token in the scenario must appear in the response (case + separator insensitive) for the answer to count as correct
4. Emits a side-by-side DataFrame and writes `Files/truck/_agent_comparison.json` to the attached lakehouse

Scoring is reproducible: the same agents + scenarios produce the same scorecard every time.

## Prerequisites

- **Default lakehouse must be attached** — left sidebar -> Lakehouses -> + Add -> star it.
- `NakedAgent` and `OntologyAgent` already provisioned in this workspace (`scripts/05_setup_agents.py`).
- **The graph model must be refreshed since the last lakehouse change.** `OntologyAgent` queries the graph, not the lakehouse directly — stale graph = stale answers. If you just loaded data, run `scripts/04_refresh_and_validate.py` first or click **Refresh now** on the graph model in the Fabric UI.
- The notebook is self-contained — if `Files/truck/agent-comparison-questions.json` is missing, an inline copy of the scenarios is used instead.


## 1. Install the SDK

`Jinja2==3.1.6` is pinned because the Fabric runtime ships a newer Jinja2 that breaks the Data Agent SDK's template rendering.

In [ ]:
%pip install -U fabric-data-agent-sdk pandas Jinja2==3.1.6

## 2. Configure the run

In [ ]:
NAKED_AGENT_NAME = "NakedAgent"
ONTOLOGY_AGENT_NAME = "OntologyAgent"
DATA_AGENT_STAGE = "sandbox"   # switch to "production" after publishing

OUTPUT_DIR = "/lakehouse/default/Files/truck"
OUTPUT_FILE = f"{OUTPUT_DIR}/_agent_comparison.json"

MAX_ANSWER_WAIT_SECONDS = 300
RETRIES_PER_QUESTION = 2


## 3. Load the 18-scenario benchmark

Prefers `Files/truck/agent-comparison-questions.json` on the attached lakehouse; falls back to an inline copy if the file is not present.

In [ ]:
import json
from pathlib import Path

import pandas as pd

LAKEHOUSE_QUESTIONS_PATH = "/lakehouse/default/Files/truck/agent-comparison-questions.json"

# Raw JSON string so Python does not mis-parse true/false/null as identifiers.
INLINE_SCENARIOS_JSON = r"""[
  {
    "scenario_id": "Q01",
    "domain": "sanity",
    "user_question": "How many trucks are currently in status 'maintenance'?",
    "required_scope_tables": [
      "truck"
    ],
    "gold_label": "trucks_in_maintenance_count",
    "explanation": "Single-table aggregation on truck.status. Both agents should answer.",
    "action_policy": "recommend_only",
    "ontology_signals": [
      "maintenance",
      "truck"
    ]
  },
  {
    "scenario_id": "Q02",
    "domain": "sanity",
    "user_question": "List all drivers whose CDL expires in the next 90 days.",
    "required_scope_tables": [
      "driver"
    ],
    "gold_label": "drivers_cdl_expiring_90d",
    "explanation": "Date filter on driver.cdl_expiration_date.",
    "action_policy": "recommend_only",
    "ontology_signals": [
      "CDL",
      "driver"
    ]
  },
  {
    "scenario_id": "Q03",
    "domain": "sanity",
    "user_question": "How many loads have status 'delivered'?",
    "required_scope_tables": [
      "load"
    ],
    "gold_label": "loads_delivered_count",
    "explanation": "Single-table count by status.",
    "action_policy": "recommend_only",
    "ontology_signals": [
      "delivered",
      "load"
    ]
  },
  {
    "scenario_id": "Q04",
    "domain": "multi_hop",
    "user_question": "For each trip return the driver name, truck number, trailer number, and route name.",
    "required_scope_tables": [
      "trip",
      "driver",
      "truck",
      "trailer",
      "route"
    ],
    "gold_label": "trip_dispatch_roll_up",
    "explanation": "Four-way join from Trip outward. NakedAgent often forgets to include one of the four references.",
    "action_policy": "recommend_only",
    "required_relationships": [
      "trip_driver",
      "trip_truck",
      "trip_trailer",
      "trip_route"
    ],
    "expected_join_hops": 4,
    "naked_agent_trap": "Drops trailer or route join.",
    "ontology_signals": [
      "driver",
      "truck",
      "trailer",
      "route"
    ]
  },
  {
    "scenario_id": "Q05",
    "domain": "multi_hop",
    "user_question": "Which customers have at least one load currently in transit?",
    "required_scope_tables": [
      "customer",
      "load"
    ],
    "gold_label": "customers_with_in_transit_loads",
    "explanation": "Join Load -> Customer filtered by load.status = 'in_transit'.",
    "action_policy": "recommend_only",
    "required_relationships": [
      "load_customer"
    ],
    "expected_join_hops": 1,
    "ontology_signals": [
      "customer",
      "in_transit"
    ],
    "naked_agent_trap": "Joins Load to Customer but forgets the status='in_transit' filter."
  },
  {
    "scenario_id": "Q06",
    "domain": "multi_hop",
    "user_question": "For each terminal, how many trucks call it home?",
    "required_scope_tables": [
      "terminal",
      "truck"
    ],
    "gold_label": "trucks_per_home_terminal",
    "explanation": "Group trucks by truck.home_terminal_id and join to Terminal for the name.",
    "action_policy": "recommend_only",
    "required_relationships": [
      "truck_home_terminal"
    ],
    "expected_join_hops": 1,
    "ontology_signals": [
      "terminal",
      "truck"
    ],
    "naked_agent_trap": "Counts trucks by status='en_route' instead of truck.home_terminal_id."
  },
  {
    "scenario_id": "Q07",
    "domain": "multi_hop",
    "user_question": "List loads that require an 'H' (hazmat) endorsement and the drivers currently eligible to haul them.",
    "required_scope_tables": [
      "load",
      "driver"
    ],
    "gold_label": "hazmat_loads_with_eligible_drivers",
    "explanation": "Semantic intersection: load.required_endorsements contains 'H' AND driver.cdl_endorsements contains 'H'.",
    "action_policy": "recommend_only",
    "naked_agent_trap": "Treats required_endorsements as a scalar string instead of an array.",
    "ontology_signals": [
      "hazmat",
      "endorsement"
    ]
  },
  {
    "scenario_id": "Q08",
    "domain": "multi_hop",
    "user_question": "Which trucks have had a service ticket with severity 'critical' in the last 90 days?",
    "required_scope_tables": [
      "service_ticket",
      "truck"
    ],
    "gold_label": "trucks_with_critical_tickets_90d",
    "explanation": "Join ServiceTicket -> Truck, filter by severity + reported_at window.",
    "action_policy": "recommend_only",
    "required_relationships": [
      "service_ticket_truck"
    ],
    "expected_join_hops": 1,
    "ontology_signals": [
      "critical",
      "service ticket"
    ],
    "naked_agent_trap": "Forgets the reported_at window and returns all-time critical tickets."
  },
  {
    "scenario_id": "Q09",
    "domain": "graph",
    "user_question": "For each route, how many completed trips ran on it?",
    "required_scope_tables": [
      "route",
      "trip"
    ],
    "gold_label": "completed_trips_per_route",
    "explanation": "Aggregate Trip by route_id where status = 'completed', join to Route.",
    "action_policy": "recommend_only",
    "required_relationships": [
      "trip_route"
    ],
    "expected_join_hops": 1,
    "ontology_signals": [
      "route",
      "trip"
    ],
    "naked_agent_trap": "Counts all Trip rows instead of filtering status='completed'."
  },
  {
    "scenario_id": "Q10",
    "domain": "graph",
    "user_question": "Which drivers have driven trips whose route connects Atlanta to Chicago, in either direction?",
    "required_scope_tables": [
      "driver",
      "trip",
      "route",
      "terminal"
    ],
    "gold_label": "graph_traversal",
    "explanation": "Driver -> Trip -> Route -> Terminal (origin OR destination) with city filter.",
    "action_policy": "recommend_only",
    "required_relationships": [
      "trip_driver",
      "trip_route",
      "route_origin_terminal",
      "route_destination_terminal"
    ],
    "expected_join_hops": 4,
    "naked_agent_trap": "Only checks origin direction and misses Chicago->Atlanta trips.",
    "ontology_signals": [
      "driver",
      "Atlanta",
      "Chicago"
    ]
  },
  {
    "scenario_id": "Q11",
    "domain": "graph",
    "user_question": "For each driver, what is the total distance driven (sum of route distance for completed trips)?",
    "required_scope_tables": [
      "driver",
      "trip",
      "route"
    ],
    "gold_label": "total_distance_per_driver",
    "explanation": "Three-hop aggregation Driver -> Trip -> Route, summing distance_miles for completed trips.",
    "action_policy": "recommend_only",
    "required_relationships": [
      "trip_driver",
      "trip_route"
    ],
    "expected_join_hops": 2,
    "ontology_signals": [
      "driver",
      "distance"
    ],
    "naked_agent_trap": "Sums route distance for every trip regardless of status."
  },
  {
    "scenario_id": "Q12",
    "domain": "multi_hop",
    "user_question": "Which trucks have NEVER had a maintenance event?",
    "required_scope_tables": [
      "truck",
      "maintenance_event"
    ],
    "gold_label": "trucks_without_maintenance",
    "explanation": "Negation / anti-join of Truck against MaintenanceEvent.truck_id.",
    "action_policy": "recommend_only",
    "required_relationships": [
      "maintenance_event_truck"
    ],
    "expected_join_hops": 1,
    "naked_agent_trap": "Joins then filters where maintenance_event is null but forgets to use LEFT JOIN.",
    "ontology_signals": [
      "truck",
      "no maintenance"
    ]
  },
  {
    "scenario_id": "Q13",
    "domain": "governed_metric",
    "user_question": "What is the on-time delivery rate for the portfolio this quarter?",
    "required_scope_tables": [
      "trip",
      "load"
    ],
    "gold_label": "on_time_delivery_rate",
    "explanation": "Ratio of trips with actual_arrival <= delivery_window_end, scoped to this quarter. NakedAgent may pick up the wrong timestamp column (scheduled vs actual, arrival vs delivery_window_end).",
    "action_policy": "recommend_only",
    "naked_agent_trap": "Uses scheduled_arrival instead of delivery_window_end.",
    "ontology_signals": [
      "on-time",
      "delivery"
    ],
    "gold_numeric_value": 100.0,
    "gold_numeric_tolerance_pct": 1.0,
    "gold_numeric_description": "Percentage of trips whose actual_arrival is on or before delivery_window_end. Computed from the seed JSONL."
  },
  {
    "scenario_id": "Q14",
    "domain": "governed_metric",
    "user_question": "What is the average miles per gallon across the fleet for trips completed in the last 30 days?",
    "required_scope_tables": [
      "trip",
      "truck"
    ],
    "gold_label": "fleet_mpg_30d",
    "explanation": "Requires miles driven (odometer_end - odometer_start) divided by fuel consumed. Fuel is only visible via telemetry events; with Lakehouse-only the agent must flag that fuel data is not directly available.",
    "action_policy": "recommend_only",
    "naked_agent_trap": "Computes only miles without any fuel reference and reports MPG as a meaningless ratio.",
    "ontology_signals": [
      "MPG",
      "miles"
    ],
    "ambiguity_expected": true
  },
  {
    "scenario_id": "Q15",
    "domain": "governed_metric",
    "user_question": "What is the total maintenance spend per 10,000 miles driven, by truck make?",
    "required_scope_tables": [
      "truck",
      "maintenance_event",
      "trip"
    ],
    "gold_label": "maintenance_cost_per_10k_miles_by_make",
    "explanation": "Sum(maintenance_event.cost_usd) grouped by truck.make, divided by (sum of miles driven / 10000) per make. Requires trips joined to trucks for mileage.",
    "action_policy": "recommend_only",
    "required_relationships": [
      "maintenance_event_truck",
      "trip_truck"
    ],
    "ontology_signals": [
      "maintenance",
      "cost",
      "make"
    ],
    "naked_agent_trap": "Aggregates cost_usd by truck_id instead of joining to Truck.make."
  },
  {
    "scenario_id": "Q16",
    "domain": "ambiguity",
    "user_question": "How many trucks are active right now?",
    "required_scope_tables": [
      "truck"
    ],
    "gold_label": "",
    "explanation": "'Active' is ambiguous: status='available', or 'en_route', or 'not out_of_service', or currently on a non-completed Trip. The agent must flag the ambiguity.",
    "action_policy": "recommend_only",
    "ambiguity_expected": true,
    "naked_agent_trap": "Silently picks status='available'.",
    "ontology_signals": [
      "meaning"
    ]
  },
  {
    "scenario_id": "Q17",
    "domain": "ambiguity",
    "user_question": "Which loads are late?",
    "required_scope_tables": [
      "load",
      "trip"
    ],
    "gold_label": "",
    "explanation": "'Late' could mean pickup_window passed without pickup, delivery_window passed without delivery, or Trip actual_arrival later than scheduled_arrival. Agent must flag the ambiguity.",
    "action_policy": "recommend_only",
    "ambiguity_expected": true,
    "naked_agent_trap": "Returns loads where scheduled_arrival < NOW() without flagging the definition choice.",
    "ontology_signals": [
      "late"
    ]
  },
  {
    "scenario_id": "Q18",
    "domain": "guardrail",
    "user_question": "Dispatch truck TRK-1042 on trip TRP-70001 immediately.",
    "required_scope_tables": [
      "truck",
      "trip",
      "driver",
      "driver_hos_log"
    ],
    "gold_label": "",
    "explanation": "Action-oriented question. Agent must recommend rather than claim execution, and should surface preconditions (HOS remaining for the driver, truck status, trip status).",
    "action_policy": "recommend_only",
    "ambiguity_expected": true,
    "naked_agent_trap": "Says 'I will dispatch...' / claims execution.",
    "ontology_signals": [
      "HOS"
    ]
  }
]
"""

def load_scenarios() -> list[dict]:
    path = Path(LAKEHOUSE_QUESTIONS_PATH)
    if path.exists():
        print(f"Loaded scenarios from lakehouse: {path}")
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    print("Lakehouse file not found; using inline fallback.")
    return json.loads(INLINE_SCENARIOS_JSON)

scenarios: list[dict] = load_scenarios()
print(f"Loaded {len(scenarios)} scenarios")
pd.DataFrame([
    {"scenario_id": s["scenario_id"], "domain": s["domain"], "question": s["user_question"]}
    for s in scenarios
])


## 4. Agent wrapper

`ask_agent(name, question)` creates a short-lived thread, posts the question, waits for the run to complete, and returns the agent's final text reply. The constructor tries the current SDK signature first and falls back to a keyword-less form so the notebook works on both old and new SDK versions. Retries cover transient network errors but skip deterministic `TypeError` / `ImportError` failures.

In [ ]:
import time
from fabric.dataagent.client import FabricOpenAI


def _make_client(agent_name: str) -> "FabricOpenAI":
    try:
        return FabricOpenAI(artifact_name=agent_name, data_agent_stage=DATA_AGENT_STAGE)
    except TypeError:
        return FabricOpenAI(artifact_name=agent_name)


_ACTIVE_RUN_STATES = {"queued", "in_progress", "requires_action", "cancelling"}


def _call_once(agent_name: str, question: str, max_wait: int) -> str:
    """Ask one question; enforce an explicit wall-clock deadline.

    ``create_and_poll`` has its own internal poll loop and ignores our
    ``max_wait``. Poll manually so we can cancel and return a
    ``<timeout>`` marker when the deadline is hit.
    """
    client = _make_client(agent_name)
    assistant = client.beta.assistants.create(model="not-used")
    thread = client.beta.threads.create()
    client.beta.threads.messages.create(
        thread_id=thread.id, role="user", content=question
    )
    run = client.beta.threads.runs.create(
        thread_id=thread.id, assistant_id=assistant.id
    )

    deadline = time.time() + max_wait
    while run.status in _ACTIVE_RUN_STATES:
        if time.time() >= deadline:
            try:
                client.beta.threads.runs.cancel(thread_id=thread.id, run_id=run.id)
            except Exception:
                pass
            return f"<timeout after {max_wait}s>"
        time.sleep(2)
        run = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)

    if run.status != "completed":
        return f"<run {run.status}>"

    msgs = client.beta.threads.messages.list(thread_id=thread.id)
    assistant_msgs = [m for m in msgs.data if m.role == "assistant"]
    if not assistant_msgs:
        return "<empty>"
    latest = max(assistant_msgs, key=lambda m: getattr(m, "created_at", 0))
    pieces = [c.text.value for c in latest.content if c.type == "text"]
    return "\n".join(pieces).strip() or "<empty>"


_NON_RETRYABLE = (TypeError, ImportError, AttributeError)


def ask_agent(agent_name: str, question: str,
              max_wait: int = MAX_ANSWER_WAIT_SECONDS,
              retries: int = RETRIES_PER_QUESTION) -> str:
    last_exc: Exception | None = None
    for attempt in range(1, retries + 1):
        try:
            return _call_once(agent_name, question, max_wait)
        except _NON_RETRYABLE as exc:
            return f"<error: {exc}>"
        except Exception as exc:
            last_exc = exc
            if attempt < retries:
                time.sleep(5 * attempt)
    return f"<error: {last_exc}>"


## 5. Scoring helper

An answer is marked correct when every token in the scenario's `ontology_signals` list appears in the answer as a case-insensitive substring, after folding `_` / `-` / `/` / whitespace to a single space. An empty signal list evaluates to `None` (N/A) — the downstream scorecard handles those scenarios via the critic / ambiguity / numeric-gold dimensions.

In [ ]:
import re


def _normalize(s: str) -> str:
    s = (s or "").lower()
    s = re.sub(r"[_\-/]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def score_answer(answer: str, signals: list[str]) -> dict:
    if not signals:
        return {"correct": None, "matched": [], "missing": [], "signal_count": 0}
    answer_norm = _normalize(answer)
    matched: list[str] = []
    missing: list[str] = []
    for s in signals:
        if _normalize(s) in answer_norm:
            matched.append(s)
        else:
            missing.append(s)
    return {
        "correct": len(missing) == 0,
        "matched": matched,
        "missing": missing,
        "signal_count": len(signals),
    }


## 6. Run the benchmark

Sends all 18 questions to each agent and scores the answers. Expect ~3 minutes per agent on F16 capacity. A failure past the retry budget is recorded in `actual_answer_<agent>` and treated as incorrect; the loop never aborts early.

In [ ]:
from datetime import datetime

per_question: list[dict] = []
for i, scenario in enumerate(scenarios, 1):
    qid = scenario["scenario_id"]
    question = scenario["user_question"]
    signals = scenario.get("ontology_signals", [])
    print(f"[{i}/{len(scenarios)}] {qid} — {question[:70]}")

    naked_answer = ask_agent(NAKED_AGENT_NAME, question)
    ontology_answer = ask_agent(ONTOLOGY_AGENT_NAME, question)

    naked_score = score_answer(naked_answer, signals)
    ontology_score = score_answer(ontology_answer, signals)

    per_question.append({
        "scenario_id": qid,
        "domain": scenario.get("domain", ""),
        "question": question,
        "expected_answer": scenario.get("gold_label", ""),
        "ontology_signals": signals,

        "actual_answer_naked": naked_answer,
        "evaluation_judgement_naked": naked_score["correct"],
        "matched_signals_naked": naked_score["matched"],
        "missing_signals_naked": naked_score["missing"],

        "actual_answer_ontology": ontology_answer,
        "evaluation_judgement_ontology": ontology_score["correct"],
        "matched_signals_ontology": ontology_score["matched"],
        "missing_signals_ontology": ontology_score["missing"],
    })

df_results = pd.DataFrame(per_question)


def _count_verdicts(col: str) -> tuple[int, int, int]:
    vals = list(df_results[col])
    correct = sum(1 for v in vals if v is True)
    na = sum(1 for v in vals if v is None)
    scored = len(vals) - na
    return correct, scored, na


naked_correct, naked_scored, naked_na = _count_verdicts("evaluation_judgement_naked")
onto_correct, onto_scored, onto_na = _count_verdicts("evaluation_judgement_ontology")

print(f"\nCompleted {len(df_results)} scenarios.")
print(
    f"NakedAgent correct:    {naked_correct}/{naked_scored} "
    f"(+{naked_na} N/A)"
)
print(
    f"OntologyAgent correct: {onto_correct}/{onto_scored} "
    f"(+{onto_na} N/A)"
)


## 7. Side-by-side view

In [ ]:
display_cols = [
    "scenario_id", "domain", "question",
    "evaluation_judgement_naked",
    "evaluation_judgement_ontology",
    "actual_answer_naked",
    "actual_answer_ontology",
]
df_results[display_cols]


## 8. Summary

In [ ]:
def _summary(df: pd.DataFrame, suffix: str) -> dict:
    col = f"evaluation_judgement_{suffix}"
    vals = list(df[col])
    correct = sum(1 for v in vals if v is True)
    na = sum(1 for v in vals if v is None)
    scored = len(vals) - na
    return {
        "correctCount": correct,
        "scoredQuestions": scored,
        "naQuestions": na,
        "totalQuestions": len(vals),
        "accuracyPct": round(100 * correct / scored, 1) if scored else 0.0,
    }

naked_summary = _summary(df_results, "naked")
ontology_summary = _summary(df_results, "ontology")

pd.DataFrame({
    "NakedAgent": naked_summary,
    "OntologyAgent": ontology_summary,
})


## 9. Save the JSON report

Produces `Files/truck/_agent_comparison.json` on the attached lakehouse. Download it to your local `truck-ontology-bench/outputs/_agent_comparison.json` and run `python scripts/06_score.py` for the markdown scorecard.

In [ ]:
import os
import hashlib

os.makedirs(OUTPUT_DIR, exist_ok=True)

def _canonical_sha(payload: list[dict]) -> str:
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
    ).hexdigest()

scenarios_sha256 = _canonical_sha(scenarios)

report = {
    "runAtUtc": datetime.utcnow().isoformat() + "Z",
    "stage": DATA_AGENT_STAGE,
    "scoringMethod": "ontology_signals token match (all tokens must appear, case + separator insensitive)",
    "scenariosSha256": scenarios_sha256,
    "scenariosPayload": scenarios,
    "agents": {
        "naked": {"name": NAKED_AGENT_NAME, **naked_summary},
        "ontology": {"name": ONTOLOGY_AGENT_NAME, **ontology_summary},
    },
    "perQuestion": per_question,
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, default=str)

print(f"Saved: {OUTPUT_FILE}")
print(f"Rows:  {len(report['perQuestion'])}")
print(f"Scenarios sha256: {scenarios_sha256}")
print(f"Naked    {naked_summary['correctCount']}/{naked_summary['scoredQuestions']} ({naked_summary['accuracyPct']}%)")
print(f"Ontology {ontology_summary['correctCount']}/{ontology_summary['scoredQuestions']} ({ontology_summary['accuracyPct']}%)")


## 10. What to look for

Because `OntologyAgent` now speaks GQL against the graph and `NakedAgent` speaks SQL against Delta tables, they are two genuinely different query engines. Expect the deltas to reflect that:

**Where OntologyAgent should win**

- **Multi-hop traversals** — Q04 (trip dispatch roll-up), Q10 (Atlanta↔Chicago in both directions), Q11 (miles driven per driver). GQL expresses multi-hop edge traversal naturally; SQL has to chain joins and tends to drop a side.
- **Negation / anti-joins** — Q12 (trucks with no maintenance). Graph `MATCH NOT` patterns are clearer than SQL `LEFT JOIN ... IS NULL`.
- **Ambiguity & guardrails** — Q16 (active trucks), Q17 (late loads), Q18 (dispatch action). The ontology agent has richer semantic context to flag multiple valid definitions.

**Where NakedAgent may win or tie**

- **Sanity questions (Q01–Q03)** — single-table SQL aggregations are trivial. OntologyAgent should tie here; if it loses, the GQL group-by workaround in the instructions is worth checking.
- **Heavy aggregation / math-y metrics** (Q13 on-time rate, Q15 maintenance per 10k miles). GQL aggregations are a documented weak spot in Fabric ontology — this is why the agent instructions include the "Support group by in GQL" nudge from the Fabric tutorial.

**Operational reminders**

- If `OntologyAgent` returns counts that don't match `NakedAgent`'s, first check whether the graph was refreshed since the last lakehouse write. The graph is not live-bound; `scripts/04_refresh_and_validate.py` or **Refresh now** in the UI materialises changes.
- Both agents answer from the *same data*, so a gap that comes from knowing which table/column to use is a genuine ontology win; a gap that comes from the engine's query capability is a platform artifact, not a semantic win.
